In [1]:
import pm4py
import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from event_log_transformer import *
pandas.set_option('display.max_columns', None)
pandas.set_option('display.max_rows', 200)


In [2]:
file_path = '../../../../data/BPI Challenge 2017.xes'
event_log = pm4py.read_xes(file_path)

parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

In [3]:
event_log

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202262,Deleted,User_1,W_Call after offers,Workflow,Workitem_1817549786,ate_abort,2017-01-06 06:33:02.212000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202263,Created,User_1,W_Call after offers,Workflow,Workitem_363876066,schedule,2017-01-06 06:33:02.221000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202264,statechange,User_28,A_Cancelled,Application,ApplState_1869071797,complete,2017-01-16 09:51:21.114000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202265,statechange,User_28,O_Cancelled,Offer,OfferState_420066181,complete,2017-01-16 09:51:21.139000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Offer_1580299144


In [4]:
event_log['lifecycle:transition'].unique()

array(['complete', 'schedule', 'withdraw', 'start', 'suspend',
       'ate_abort', 'resume'], dtype=object)

In [5]:
start_end_event_log2 = TransformEventLog.start_end_event_log_all(event_log
                                                                )
start_end_event_log2 = TransformEventLog.seconds_in_day(start_end_event_log2, 'time:timestamp_start')
start_end_event_log2 = TransformEventLog.day_of_week(start_end_event_log2, 'time:timestamp_start')

In [6]:
start_end_event_log2.groupby(['lifecycle:transition_start', 'lifecycle:transition_complete']).count()

Action_start  \
lifecycle:transition_start lifecycle:transition_complete                 
ate_abort                  schedule                              18834   
complete                   complete                              64582   
                           schedule                               9697   
resume                     complete                              21229   
                           start                                   815   
                           suspend                              105116   
schedule                   start                                127210   
                           withdraw                              21844   
start                      complete                              20633   
                           start                                   202   
                           suspend                              107392   
suspend                    ate_abort                             85224   
                           resume                               127160   
                           suspend                                2894   
withdraw                   schedule                                 27   

                                                          org:resource_start  \
lifecycle:transition_start lifecycle:transition_complete                       
ate_abort                  schedule                                    18834   
complete                   complete                                    64582   
                           schedule                                     9697   
resume                     complete                                    21229   
                           start                                         815   
                           suspend                                    105116   
schedule                   start                                      127210   
                           withdraw                                    21844   
start                      complete                                    20633   
                           start                                         202   
                           suspend                                    107392   
suspend                    ate_abort                                   85224   
                           resume                                     127160   
                           suspend                                      2894   
withdraw                   schedule                                       27   

                                                          concept:name  \
lifecycle:transition_start lifecycle:transition_complete                 
ate_abort                  schedule                              18834   
complete                   complete                              64582   
                           schedule                               9697   
resume                     complete                              21229   
                           start                                   815   
                           suspend                              105116   
schedule                   start                                127210   
                           withdraw                              21844   
start                      complete                              20633   
                           start                                   202   
                           suspend                              107392   
suspend                    ate_abort                             85224   
                           resume                               127160   
                           suspend                                2894   
withdraw                   schedule                                 27   

                                                          EventOrigin_start  \
lifecycle:transition_start lifecycle:transition_complete                      
ate_abort            

In [7]:
start_end_event_log2[((start_end_event_log2['lifecycle:transition_start'] == 'complete')
                      & (start_end_event_log2['lifecycle:transition_complete'] == 'complete')
                      & (start_end_event_log2['concept:name'].str.startswith('W')))]

# ALL complete -> complete events are not activities but events

,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,case:RequestedAmount_start,FirstWithdrawalAmount_start,NumberOfTerms_start,Accepted_start,MonthlyCost_start,Selected_start,CreditScore_start,OfferedAmount_start,OfferID_start,Action_complete,org:resource_complete,EventOrigin_complete,EventID_complete,lifecycle:transition_complete,time:timestamp_complete,case:LoanGoal_complete,case:ApplicationType_complete,case:RequestedAmount_complete,FirstWithdrawalAmount_complete,NumberOfTerms_complete,Accepted_complete,MonthlyCost_complete,Selected_complete,CreditScore_complete,OfferedAmount_complete,OfferID_complete,duration,duration_seconds,duration_ms,duration_hours,seconds_in_day,day_of_week


We remove the events

In [8]:
start_end_event_log2 = start_end_event_log2[~((start_end_event_log2['lifecycle:transition_start'] == 'complete')
                      & (start_end_event_log2['lifecycle:transition_complete'] == 'complete'))]

In [9]:
start_end_event_log2.groupby(['lifecycle:transition_start', 'lifecycle:transition_complete']).count()

Action_start  \
lifecycle:transition_start lifecycle:transition_complete                 
ate_abort                  schedule                              18834   
complete                   schedule                               9697   
resume                     complete                              21229   
                           start                                   815   
                           suspend                              105116   
schedule                   start                                127210   
                           withdraw                              21844   
start                      complete                              20633   
                           start                                   202   
                           suspend                              107392   
suspend                    ate_abort                             85224   
                           resume                               127160   
                           suspend                                2894   
withdraw                   schedule                                 27   

                                                          org:resource_start  \
lifecycle:transition_start lifecycle:transition_complete                       
ate_abort                  schedule                                    18834   
complete                   schedule                                     9697   
resume                     complete                                    21229   
                           start                                         815   
                           suspend                                    105116   
schedule                   start                                      127210   
                           withdraw                                    21844   
start                      complete                                    20633   
                           start                                         202   
                           suspend                                    107392   
suspend                    ate_abort                                   85224   
                           resume                                     127160   
                           suspend                                      2894   
withdraw                   schedule                                       27   

                                                          concept:name  \
lifecycle:transition_start lifecycle:transition_complete                 
ate_abort                  schedule                              18834   
complete                   schedule                               9697   
resume                     complete                              21229   
                           start                                   815   
                           suspend                              105116   
schedule                   start                                127210   
                           withdraw                              21844   
start                      complete                              20633   
                           start                                   202   
                           suspend                              107392   
suspend                    ate_abort                             85224   
                           resume                               127160   
                           suspend                                2894   
withdraw                   schedule                                 27   

                                                          EventOrigin_start  \
lifecycle:transition_start lifecycle:transition_complete                      
ate_abort                  schedule                                   18834   
complete                   schedule                                    9697   
resume                     complete                                   21229   
            

The dataset has some errors, e.g., a series of 'suspend' events for a single activity which is noncompliant

- We remove them

In [10]:
start_end_event_log2[((start_end_event_log2['lifecycle:transition_start'] == 'suspend')
                      & (start_end_event_log2['lifecycle:transition_complete'] == 'suspend'))]

,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,case:RequestedAmount_start,FirstWithdrawalAmount_start,NumberOfTerms_start,Accepted_start,MonthlyCost_start,Selected_start,CreditScore_start,OfferedAmount_start,OfferID_start,Action_complete,org:resource_complete,EventOrigin_complete,EventID_complete,lifecycle:transition_complete,time:timestamp_complete,case:LoanGoal_complete,case:ApplicationType_complete,case:RequestedAmount_complete,FirstWithdrawalAmount_complete,NumberOfTerms_complete,Accepted_complete,MonthlyCost_complete,Selected_complete,CreditScore_complete,OfferedAmount_complete,OfferID_complete,duration,duration_seconds,duration_ms,duration_hours,seconds_in_day,day_of_week
6433088,Released,User_1,W_Validate application,Workflow,Workitem_1000829148,suspend,2016-10-10 22:00:01.615000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Released,User_1,Workflow,Workitem_452678591,suspend,2016-10-10 22:00:01.619000+00:00,Existing loan takeover,New credit,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:00:00.004000,0.004,4.0,1.111111e-06,79201,0
231997,Released,User_1,W_Call after offers,Workflow,Workitem_1001105137,suspend,2016-01-12 23:00:24.746000+00:00,Home improvement,Limit raise,Application_454240699,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Released,User_1,Workflow,Workitem_222175618,suspend,2016-01-12 23:00:24.761000+00:00,Home improvement,Limit raise,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:00:00.015000,0.015,15.0,4.166667e-06,82824,1
7847823,Released,User_1,W_Call incomplete files,Workflow,Workitem_1001722953,suspend,2016-12-05 23:00:01.720000+00:00,Unknown,New credit,Application_1458413157,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Released,User_1,Workflow,Workitem_1479926446,suspend,2016-12-05 23:00:01.724000+00:00,Unknown,New credit,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:00:00.004000,0.004,4.0,1.111111e-06,82801,0
7184420,Released,User_1,W_Assess potential fraud,Workflow,Workitem_1001917218,suspend,2016-12-12 23:00:01.970000+00:00,Home improvement,New credit,Application_483126363,7500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Released,User_1,Workflow,Workitem_2083853124,suspend,2016-12-12 23:00:01.977000+00:00,Home improvement,New credit,7500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:00:00.007000,0.007,7.0,1.944444e-06,82801,0
8416250,Released,User_1,W_Validate application,Workflow,Workitem_1002019346,suspend,2017-01-04 23:00:02.528000+00:00,Car,New credit,Application_315713743,7000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Released,User_1,Workflow,Workitem_2029601152,suspend,2017-01-04 23:00:02.547000+00:00,Car,New credit,7000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:00:00.019000,0.019,19.0,5.277778e-06,82802,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5261413,Released,User_1,W_Call incomplete files,Workflow,Workitem_997092451,suspend,2016-08-19 22:00:08.975000+00:00,Unknown,New credit,Application_2103564840,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Released,User_1,Workflow,Workitem_1845114807,suspend,2016-08-19 22:00:08.980000+00:00,Unknown,New credit,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:00:00.005000,0.005,5.0,1.388889e-06,79208,4
2479763,Released,User_1,W_Call after offers,Workflow,Workitem_997115539,suspend,2016-04-23 22:00:09.951000+00:00,Existing loan takeover,New credit,Application_831994177,22000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Released,User_1,Workflow,Workitem_328524643,suspend,2016-04-23 22:00:09.965000+00:00,Existing loan takeover,New credit,22000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:00:00.014000,0.014,14.0,3.888889e-06,79209,5
2741538,Released,User_1,W_Complete application,Workflow,Workitem_998369055,suspend,2016-05-04 22:00:23.793000+00:00,Home improvement,New credit,Applic

In [11]:
event_log[event_log['case:concept:name'] == 'Application_1270478188']

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
894254,Created,User_1,A_Create Application,Application,Application_1270478188,complete,2016-09-29 20:46:24.044000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894255,statechange,User_1,A_Submitted,Application,ApplState_1623559737,complete,2016-09-29 20:46:24.902000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894256,Created,User_1,W_Handle leads,Workflow,Workitem_209843805,schedule,2016-09-29 20:46:25.041000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894257,Deleted,User_1,W_Handle leads,Workflow,Workitem_1949416873,withdraw,2016-09-29 20:47:29.430000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894258,Created,User_1,W_Complete application,Workflow,Workitem_127594403,schedule,2016-09-29 20:47:29.436000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894259,statechange,User_1,A_Concept,Application,ApplState_1244034559,complete,2016-09-29 20:47:29.441000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894260,Obtained,User_104,W_Complete application,Workflow,Workitem_680465962,start,2016-09-30 07:23:50.829000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894261,Released,User_104,W_Complete application,Workflow,Workitem_2089063023,suspend,2016-09-30 07:27:52.511000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894262,statechange,User_48,A_Accepted,Application,ApplState_8186816,complete,2016-09-30 15:01:59.803000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
894263,Created,User_48,O_Create Offer,Offer,Offer_446585442,complete,2016-09-30 15:03:41.761000+00:00,Existing loan takeover,New credit,Application_1270478188,15000.0,0.0,89.0,True,200.0,True,768.0,15000.0,NaN


In [12]:
start_end_event_log2 = start_end_event_log2[~start_end_event_log2['case:concept:name'].isin(
    start_end_event_log2[start_end_event_log2['lifecycle:transition_start'] == start_end_event_log2['lifecycle:transition_complete']]['case:concept:name'].values
)]

In [13]:
start_end_event_log2.groupby(['lifecycle:transition_start', 'lifecycle:transition_complete']).count()

Action_start  \
lifecycle:transition_start lifecycle:transition_complete                 
ate_abort                  schedule                              18042   
complete                   schedule                               9238   
resume                     complete                              20335   
                           start                                   696   
                           suspend                              100296   
schedule                   start                                122781   
                           withdraw                              21226   
start                      complete                              19989   
                           suspend                              103488   
suspend                    ate_abort                             82340   
                           resume                               121327   
withdraw                   schedule                                 25   

                                                          org:resource_start  \
lifecycle:transition_start lifecycle:transition_complete                       
ate_abort                  schedule                                    18042   
complete                   schedule                                     9238   
resume                     complete                                    20335   
                           start                                         696   
                           suspend                                    100296   
schedule                   start                                      122781   
                           withdraw                                    21226   
start                      complete                                    19989   
                           suspend                                    103488   
suspend                    ate_abort                                   82340   
                           resume                                     121327   
withdraw                   schedule                                       25   

                                                          concept:name  \
lifecycle:transition_start lifecycle:transition_complete                 
ate_abort                  schedule                              18042   
complete                   schedule                               9238   
resume                     complete                              20335   
                           start                                   696   
                           suspend                              100296   
schedule                   start                                122781   
                           withdraw                              21226   
start                      complete                              19989   
                           suspend                              103488   
suspend                    ate_abort                             82340   
                           resume                               121327   
withdraw                   schedule                                 25   

                                                          EventOrigin_start  \
lifecycle:transition_start lifecycle:transition_complete                      
ate_abort                  schedule                                   18042   
complete                   schedule                                    9238   
resume                     complete                                   20335   
                           start                                        696   
                           suspend                                   100296   
schedule                   start                                     122781   
                           withdraw                                   21226   
start                      complete                                   19989   
                           suspend                                   1034

In [14]:
start_end_event_log2['case:concept:name'].unique().shape

(30609,)

In [15]:
event_log['case:concept:name'].unique().shape

(31509,)

In [16]:
start_end_event_log2.shape

(619783, 42)

In [17]:
# add lifecycle:transition_start to concept:name

In [18]:
start_end_event_log2['concept:name'] = start_end_event_log2['concept:name'] + '__' + start_end_event_log2['lifecycle:transition_start']

In [19]:
start_end_event_log2 = TransformEventLog.value_count_per_case(start_end_event_log2, 'org:resource_start',
                                                                  timestamp_name = 'time:timestamp_start',
                                                                 lifecycle_col_name = 'lifecycle:transition_start')

/home/LordKunkler/TaskExecutionTimeMining/src/notebooks/standard/BPIC_2017/../../../TaskExecutionTimeMining/event_log_transformer.py:187: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  value_count_event_log = value_count_event_log.fillna(0)


In [20]:
start_end_event_log2 = TransformEventLog.value_count_per_case(start_end_event_log2, 'concept:name',
                                                                  timestamp_name = 'time:timestamp_start',
                                                                 lifecycle_col_name = 'lifecycle:transition_start' )

In [21]:
start_end_event_log2 = start_end_event_log2.sort_values(by='time:timestamp_start')

In [22]:
start_end_event_log2

,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,case:RequestedAmount_start,FirstWithdrawalAmount_start,NumberOfTerms_start,Accepted_start,MonthlyCost_start,Selected_start,CreditScore_start,OfferedAmount_start,OfferID_start,Action_complete,org:resource_complete,EventOrigin_complete,EventID_complete,lifecycle:transition_complete,time:timestamp_complete,case:LoanGoal_complete,case:ApplicationType_complete,case:RequestedAmount_complete,FirstWithdrawalAmount_complete,NumberOfTerms_complete,Accepted_complete,MonthlyCost_complete,Selected_complete,CreditScore_complete,OfferedAmount_complete,OfferID_complete,duration,duration_seconds,duration_ms,duration_hours,seconds_in_day,day_of_week,User_1,User_10,User_100,User_101,User_102,User_103,User_104,User_105,User_106,User_107,User_108,User_109,User_11,User_110,User_111,User_112,User_113,User_114,User_115,User_116,User_117,User_118,User_119,User_12,User_120,User_121,User_122,User_123,User_124,User_125,User_126,User_127,User_128,User_129,User_13,User_130,User_131,User_132,User_133,User_134,User_135,User_136,User_137,User_138,User_139,User_14,User_140,User_141,User_142,User_143,User_144,User_145,User_146,User_147,User_148,User_149,User_15,User_16,User_17,User_18,User_19,User_2,User_20,User_21,User_22,User_23,User_24,User_25,User_26,User_27,User_28,User_29,User_3,User_30,User_31,User_32,User_33,User_34,User_35,User_36,User_37,User_38,User_39,User_4,User_40,User_41,User_42,User_43,User_44,User_45,User_46,User_47,User_48,User_49,User_5,User_50,User_51,User_52,User_53,User_54,User_55,User_56,User_57,User_58,User_59,User_6,User_60,User_61,User_62,User_63,User_64,User_65,User_66,User_67,User_68,User_69,User_7,User_70,User_71,User_72,User_73,User_74,User_75,User_76,User_77,User_78,User_79,User_8,User_80,User_81,User_82,User_83,User_84,User_85,User_86,User_87,User_88,User_89,User_9,User_90,User_91,User_92,User_93,User_94,User_95,User_96,User_97,User_98,User_99,W_Assess potential fraud__ate_abort,W_Assess potential fraud__complete,W_Assess potential fraud__resume,W_Assess potential fraud__schedule,W_Assess potential fraud__start,W_Assess potential fraud__suspend,W_Assess potential fraud__withdraw,W_Call after offers__ate_abort,W_Call after offers__complete,W_Call after offers__resume,W_Call after offers__schedule,W_Call after offers__start,W_Call after offers__suspend,W_Call after offers__withdraw,W_Call incomplete files__ate_abort,W_Call incomplete files__complete,W_Call incomplete files__resume,W_Call incomplete files__schedule,W_Call incomplete files__start,W_Call incomplete files__suspend,W_Complete application__ate_abort,W_Complete application__complete,W_Complete application__resume,W_Complete application__schedule,W_Complete application__start,W_Complete application__suspend,W_Handle leads__complete,W_Handle leads__resume,W_Handle leads__schedule,W_Handle leads__start,W_Handle leads__suspend,W_Handle leads__withdraw,W_Shortened completion __resume,W_Shortened completion __schedule,W_Shortened completion __start,W_Shortened completion __suspend,W_Validate application__ate_abort,W_Validate application__complete,W_Validate application__resume,W_Validate application__schedule,W_Validate application__start,W_Validate application__suspend
96037,Created,User_1,W_Handle leads__schedule,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,0.0,0.0,0,0.0,0,0.0,0.0,0,Deleted,User_1,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,20000.0,0.0,0.0,0,0.0,0,0.0,0.0,0,0 days 00:01:20.618000,80.618,80618.0,2.239389e-02,35475,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

In [23]:
train, test = pm4py.ml.split_train_test(start_end_event_log2)

In [24]:
print(train.shape, test.shape)
print(train['case:concept:name'].unique().shape, test['case:concept:name'].unique().shape)

(495875, 233) (123908, 233)
(24475,) (6134,)


In [26]:
train.to_csv('../../transformed_event_logs/BPIC_2017_all_train.csv', index=False, date_format='%Y-%m-%d %H:%M:%S.%f')
train.to_pickle('../../transformed_event_logs/BPIC_2017_all_train.pickle')

In [27]:
test.to_csv('../../transformed_event_logs/BPIC_2017_all_test.csv', index=False, date_format='%Y-%m-%d %H:%M:%S.%f')
test.to_pickle('../../transformed_event_logs/BPIC_2017_all_test.pickle')